# Apache Airflow — ML Pipeline Orchestration

Airflow is the industry standard for orchestrating complex data and ML workflows. DAGs (Directed Acyclic Graphs) define dependencies between tasks, enabling scheduled, monitored, and alertable pipelines.

**Topics:** DAG structure, operators, sensors, dynamic DAGs, ML training pipelines, monitoring & alerting

In [ ]:
# Install: pip install apache-airflow apache-airflow-providers-amazon
# Quickstart: airflow standalone (starts webserver + scheduler)

print("""
=== Airflow Setup ===

Installation:
  pip install apache-airflow==2.8.0
  
  # With providers for cloud services
  pip install apache-airflow-providers-amazon   # AWS S3, SageMaker
  pip install apache-airflow-providers-google   # GCS, BigQuery, Vertex AI
  pip install apache-airflow-providers-slack    # Slack alerts

Quickstart (development):
  export AIRFLOW_HOME=~/airflow
  airflow standalone  # starts webserver + scheduler
  # Access UI at http://localhost:8080

Docker Compose (production-like):
  curl -LfO https://airflow.apache.org/docs/apache-airflow/stable/docker-compose.yaml
  docker compose up -d
""")

## 1. Complete ML Training DAG

In [ ]:
ml_dag = '''
# dags/ml_training_pipeline.py
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.bash import BashOperator
from airflow.operators.email import EmailOperator
from airflow.sensors.filesystem import FileSensor
from airflow.utils.dates import days_ago
from airflow.models import Variable
from datetime import datetime, timedelta
import logging

logger = logging.getLogger(__name__)

# Default args applied to all tasks
default_args = {
    'owner': 'ml-team',
    'depends_on_past': False,
    'email': ['ml-alerts@company.com'],
    'email_on_failure': True,
    'email_on_retry': False,
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
    'retry_exponential_backoff': True,
    'execution_timeout': timedelta(hours=2),
}

with DAG(
    dag_id='ml_training_pipeline',
    description='Weekly credit risk model retraining',
    default_args=default_args,
    schedule_interval='0 2 * * 1',  # Every Monday at 2 AM
    start_date=days_ago(7),
    catchup=False,
    tags=['ml', 'credit-risk', 'weekly'],
    max_active_runs=1,  # prevent concurrent runs
) as dag:
    
    # --- Task functions ---
    
    def extract_data(**context):
        """Extract training data from data warehouse."""
        from datetime import datetime, timedelta
        run_date = context[\'execution_date\']
        start_date = run_date - timedelta(days=90)
        logger.info(f"Extracting data from {start_date} to {run_date}")
        # ... SQL query to BigQuery/Snowflake ...
        n_rows = 150_000  # simulated
        # Push result to XCom for downstream tasks
        context[\'task_instance\'].xcom_push(key=\'n_rows\', value=n_rows)
        return n_rows
    
    def validate_data(**context):
        """Validate data quality before training."""
        n_rows = context[\'task_instance\'].xcom_pull(key=\'n_rows\', task_ids=\'extract_data\')
        assert n_rows > 10_000, f"Too few rows: {n_rows}"
        logger.info(f"Data validation passed: {n_rows} rows")
    
    def train_model(**context):
        """Train model and log to MLflow."""
        import mlflow
        mlflow.set_tracking_uri(Variable.get(\'mlflow_tracking_uri\'))
        with mlflow.start_run(run_name=f"weekly_{context[\'ds\']}") as run:
            # ... training logic ...
            mlflow.log_metric(\'val_auc\', 0.924)
            context[\'task_instance\'].xcom_push(key=\'run_id\', value=run.info.run_id)
            context[\'task_instance\'].xcom_push(key=\'val_auc\', value=0.924)
    
    def check_performance(**context):
        """Branch: promote model if AUC improves."""
        val_auc = context[\'task_instance\'].xcom_pull(key=\'val_auc\', task_ids=\'train_model\')
        threshold = float(Variable.get(\'min_auc_threshold\', default_var=\'0.90\'))
        if val_auc >= threshold:
            return \'promote_model\'  # go to promote task
        else:
            return \'alert_degradation\'  # go to alert task
    
    def promote_model(**context):
        """Register and promote model to staging."""
        run_id = context[\'task_instance\'].xcom_pull(key=\'run_id\', task_ids=\'train_model\')
        # mlflow.register_model(...)
        logger.info(f"Model {run_id} promoted to staging")
    
    # --- Task definitions ---
    
    check_new_data = FileSensor(
        task_id=\'check_new_data\',
        filepath=\'/data/new_batch_*.parquet\',
        poke_interval=300,  # check every 5 minutes
        timeout=3600,       # fail after 1 hour
        mode=\'reschedule\',  # release worker slot while waiting
    )
    
    extract = PythonOperator(task_id=\'extract_data\', python_callable=extract_data)
    validate = PythonOperator(task_id=\'validate_data\', python_callable=validate_data)
    train = PythonOperator(task_id=\'train_model\', python_callable=train_model)
    
    check_perf = BranchPythonOperator(
        task_id=\'check_performance\', python_callable=check_performance
    )
    
    promote = PythonOperator(task_id=\'promote_model\', python_callable=promote_model)
    
    alert = EmailOperator(
        task_id=\'alert_degradation\',
        to=[\'ml-team@company.com\'],
        subject=\'Model Performance Degradation\',
        html_content=\'AUC below threshold. Manual review required.\',
    )
    
    # --- Task dependencies (DAG structure) ---
    # check_new_data → extract → validate → train → check_perf
    #                                                    ↓           ↓
    #                                                promote     alert
    
    check_new_data >> extract >> validate >> train >> check_perf >> [promote, alert]
'''

print('=== ML Training DAG ===')
print(ml_dag)

## 2. Dynamic DAGs for Parallel Model Training

In [ ]:
dynamic_dag = '''
# dags/parallel_hyperparameter_search.py
# Dynamically create tasks for parallel hyperparameter search

from airflow.decorators import dag, task
from datetime import datetime
import itertools

PARAM_GRID = {
    \'n_estimators\': [50, 100, 200],
    \'max_depth\': [3, 5, 7],
    \'learning_rate\': [0.05, 0.1],
}

# All combinations = 3×3×2 = 18 parallel training jobs
param_combinations = [
    dict(zip(PARAM_GRID.keys(), vals))
    for vals in itertools.product(*PARAM_GRID.values())
]

@dag(
    dag_id=\'parallel_hparam_search\',
    schedule_interval=None,  # triggered manually
    start_date=datetime(2024, 1, 1),
    catchup=False,
)
def create_hparam_dag():
    
    @task
    def prepare_data():
        """Load and prepare dataset (runs once)."""
        # ... load data ...
        return {\'n_train\': 10000, \'n_test\': 2000}
    
    @task
    def train_one_config(params: dict, data_info: dict) -> dict:
        """Train model with specific hyperparameters."""
        from sklearn.ensemble import GradientBoostingClassifier
        import numpy as np
        # ... train with params ...
        auc = np.random.uniform(0.88, 0.96)  # simulated
        return {\'params\': params, \'auc\': auc}
    
    @task
    def select_best(results: list) -> dict:
        """Select best configuration."""
        best = max(results, key=lambda x: x[\'auc\'])
        print(f"Best: {best[\'params\']} → AUC={best[\'auc\']: .4f}")
        return best
    
    data_info = prepare_data()
    
    # Fan out: create 18 parallel training tasks
    results = [
        train_one_config(params=combo, data_info=data_info)
        for combo in param_combinations
    ]
    
    # Fan in: collect results
    best = select_best(results)

dag_instance = create_hparam_dag()
'''
print('=== Dynamic DAG for Parallel Hyperparameter Search ===')
print(dynamic_dag)

print("""
Airflow key concepts:
  DAG       — Python file defining workflow; stored in dags/ directory
  Task      — individual unit of work (PythonOperator, BashOperator, etc.)
  XCom      — cross-task communication (small values only!)
  Sensor    — waits for external condition (file, S3 object, API)
  Hook      — connection to external service (S3Hook, PostgresHook)
  Variable  — Airflow config store (access via Variable.get())
  Connection — stored credentials (PostgreSQL, AWS, GCP)

Execution types:
  LocalExecutor      — parallel tasks on single machine
  CeleryExecutor     — distributed with Redis/RabbitMQ worker queue
  KubernetesExecutor — each task runs in its own k8s pod (scales to zero!)
"""
)